In [1]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 28.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 62.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import transformers
print(transformers.__version__)


5.5.4


In [3]:
import pandas as pd
import ast
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Charger le dataset
dataset = load_dataset("itsanmolgupta/mimic-cxr-dataset", split="train")

README.md:   0%|          | 0.00/357 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/396M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30633 [00:00<?, ? examples/s]

In [4]:
diseases = ['Atelectasis', 'Consolidation', 'Infiltration', 'Pneumothorax',
            'Edema', 'Emphysema', 'Fibrosis', 'Effusion', 'Pneumonia',
            'Pleural_Thickening', 'Cardiomegaly', 'Nodule', 'Mass', 'Hernia']

def add_probas(example):
    rapport = example['impression']
    rapport_lower = rapport.lower()
    probas = []
    for d in diseases:
        if d.lower() in rapport_lower:
            probas.append(1.0)
        else:
            probas.append(0.0)
    input_text = " ".join([f"{d}: {p:.1f}" for d, p in zip(diseases, probas)])
    example['input_text'] = input_text
    example['target_text'] = rapport
    return example

# Filtrer les rapports vides
dataset = dataset.filter(lambda x: x['impression'] is not None and x['impression'].strip() != '')

# Appliquer la transformation
dataset = dataset.map(add_probas)

Filter:   0%|          | 0/30633 [00:00<?, ? examples/s]

Map:   0%|          | 0/30623 [00:00<?, ? examples/s]

In [5]:
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
test_dataset = dataset['test']

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

Train: 27560, Test: 3063


In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
def preprocess(example):
    inputs = tokenizer(example['input_text'], truncation=True, padding='max_length', max_length=128)
    targets = tokenizer(example['target_text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = targets['input_ids']
    return inputs

tokenized_train = train_dataset.map(preprocess, batched=True)
tokenized_test = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/27560 [00:00<?, ? examples/s]

Map:   0%|          | 0/3063 [00:00<?, ? examples/s]

In [8]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir='./results_mimic',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.948140,0.832278
2,0.842538,0.768526
3,0.789298,0.741055
4,0.764725,0.725910
5,0.756012,0.721700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8615, training_loss=0.8609037110366865, metrics={'train_runtime': 2031.1166, 'train_samples_per_second': 67.844, 'train_steps_per_second': 4.242, 'total_flos': 4662525060710400.0, 'train_loss': 0.8609037110366865, 'epoch': 5.0})

In [10]:
import random

# Prendre 5 exemples aléatoires du test set
indices = random.sample(range(len(test_dataset)), 5)

for idx in indices:
    exemple = test_dataset[idx]
    # Reconstruire les probas factices
    probas_exemple = [1.0 if d.lower() in exemple['target_text'].lower() else 0.0 for d in diseases]
    
    print("=" * 50)
    print("Vrai rapport :", exemple['target_text'])
    print("Rapport généré :", generer_rapport(probas_exemple))
    print()

Vrai rapport : Ill-defined opacity in the left upper-mid lung, changing its shape and size since yesterday is mostly chest wall collection, rather than pleural effusion. Right perihilar consolidation has progressed, given status post bronchoscopy (as provided by Dr. during discussion) this is mostly aspiration. Dr. discussed the findings with Dr. m.
Rapport généré : No focal consolidation. Small left pleural effusion.

Vrai rapport : Nasogastric tube terminates in the stomach. Slight interval worsening of bilateral, multifocal areas of consolidation.
Rapport généré : No focal consolidation.

Vrai rapport : Essentially unchanged exam aside from minimal increase in linear right mid to lower lung atelectasis.
Rapport généré : Low lung volumes with bibasilar atelectasis.

Vrai rapport : No acute intrathoracic abnormalities identified. The ET tube terminates appropriately above the carina.
Rapport généré : No acute cardiopulmonary process.

Vrai rapport : No acute cardiopulmonary process.
R

In [11]:
# Sauvegarder le modèle fine‑tuné
model.save_pretrained('/kaggle/working/t5_mimic_finetuned')
tokenizer.save_pretrained('/kaggle/working/t5_mimic_finetuned')

print("Modèle sauvegardé dans /kaggle/working/t5_mimic_finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans /kaggle/working/t5_mimic_finetuned


In [12]:
from IPython.display import FileLink

# Créer une archive zip (Kaggle ne permet pas de télécharger un dossier directement)
import shutil
shutil.make_archive('t5_mimic_finetuned', 'zip', '/kaggle/working/t5_mimic_finetuned')

# Afficher le lien de téléchargement
FileLink('t5_mimic_finetuned.zip')

/kaggle/working/t5_mimic_finetuned.zip